In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance alpaca-py pyarrow')

    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Alpha Factor Generation 2 - High-Frequency Technical Factors

This notebook generates **advanced alpha factors** from high-frequency intraday trading data, focusing on **market microstructure** and **technical patterns**.

Generate a comprehensive library of **high-frequency alpha factors** covering:
- **Time-based patterns** (opening/closing sessions)
- **Technical indicators** (skewness, volatility, correlations)  
- **Volume microstructure** (clustering, amount-weighted returns)
- **Advanced statistics** (entropy, weighted moments, difference patterns)


- **Input**: Minute-level OHLCV data from datamin2 + datamin3
- **Output**: Daily factor matrices aligned with return data
- **Target**: `../data/factors/obtained_features/`



In [ ]:
# === 1. Import Libraries & Load High-Frequency Data via Alpaca ===

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import time
import warnings
warnings.filterwarnings('ignore')

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

API_KEY_ID     = os.environ.get('ALPACA_API_KEY',    'AKXDM3YDOKE9EJDBQWZC')
API_SECRET_KEY = os.environ.get('ALPACA_SECRET_KEY', 'duaAQYetNb5nJ5gSRXgbvmjU0cEkSEJGjZiupMiE')
client = StockHistoricalDataClient(API_KEY_ID, API_SECRET_KEY)

CACHE_PATH = '../data/raw/alpaca_minute_data.parquet'

if os.path.exists(CACHE_PATH):
    print(f'Loading from cache: {CACHE_PATH}')
    result_df = pd.read_parquet(CACHE_PATH)
else:
    # Fetch if cache not present (same params as Data Preparation notebook)
    SYMBOLS = [
        'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'NVDA', 'TSLA', 'JPM',
        'JNJ',  'V',    'PG',   'UNH',   'HD',   'MA',   'BAC',  'ADBE',
        'NFLX', 'XOM',  'INTC', 'AMD',   'CSCO', 'PFE',  'WMT',  'CRM',
        'ABT',  'CVX',  'NKE',  'MRK',   'COST', 'ACN',  'LLY',  'TMO',
        'ABBV', 'AVGO', 'QCOM', 'TXN',   'NEE',  'HON',  'MDT',  'UNP',
        'LOW',  'PM',   'UPS',  'BMY',   'GS',   'MS',   'BLK',  'SCHW',
        'SPGI', 'ICE',
    ]
    print('Fetching from Alpaca...')
    req = StockBarsRequest(
        symbol_or_symbols=SYMBOLS,
        timeframe=TimeFrame.Minute,
        start='2022-07-01',
        end='2023-12-31',
        feed='iex',
    )
    bars = client.get_stock_bars(req)
    raw = bars.df.reset_index()
    raw = raw.rename(columns={'symbol': 'order_book_id', 'timestamp': 'datetime'})
    if raw['datetime'].dt.tz is not None:
        raw['datetime'] = (raw['datetime']
                           .dt.tz_convert('America/New_York')
                           .dt.tz_localize(None))
    vwap_col = 'vwap' if 'vwap' in raw.columns else 'close'
    raw['money'] = raw[vwap_col] * raw['volume']
    import datetime as _dt
    t = raw['datetime'].dt.time
    raw = raw[(t >= _dt.time(9, 30)) & (t <= _dt.time(16, 0))].copy()
    result_df = raw[[
        'order_book_id', 'datetime', 'open', 'high', 'low', 'close', 'volume', 'money',
    ]].sort_values(['order_book_id', 'datetime']).reset_index(drop=True)
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    result_df.to_parquet(CACHE_PATH, index=False)
    print(f'Cached to {CACHE_PATH}')

result_df['datetime'] = pd.to_datetime(result_df['datetime'])
result_df['date']     = result_df['datetime'].dt.date

print(f'✅ Loaded {len(result_df):,} rows | '
      f'{result_df["order_book_id"].nunique()} symbols | '
      f'{result_df["date"].min()} → {result_df["date"].max()}')
print(result_df.head())


In [ ]:
# === 2. Verify Data is Ready ===
# (Data is already combined and sorted from the Alpaca fetch above.)
print(f'Dataset shape : {result_df.shape}')
print(f'Symbols       : {result_df["order_book_id"].nunique()}')
print(f'Date range    : {result_df["date"].min()} → {result_df["date"].max()}')
print(f'Columns       : {list(result_df.columns)}')


In [ ]:
# === 3. Setup Data Alignment and Export Configuration ===

print("🔧 Setting up data alignment and export configuration...")

# Load return data for date alignment
ret_path = '../data/processed/wide_data_preparation/vwap1pct_daily_data.csv'

try:
    ret_wide = pd.read_csv(ret_path, index_col=0).fillna(0)
    ret_wide.index = pd.to_datetime(ret_wide.index)
    
    print(f"✅ Return data loaded for alignment:")
    print(f"   - Shape: {ret_wide.shape}")
    print(f"   - Date range: {ret_wide.index[0]} to {ret_wide.index[-1]}")
    
    # Create date-to-index mapping for consistent factor export
    date_to_index = {}
    for i in range(len(ret_wide.index)):
        date_to_index[ret_wide.index[i]] = i
    
    print(f"📅 Date mapping created: {len(date_to_index)} trading days")
    
    # Create output directory
    output_dir = "../data/factors/obtained_features"
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output directory ready: {output_dir}")
    
except FileNotFoundError:
    print(f"❌ Error: Return data not found at {ret_path}")
    print("Factor alignment may not work correctly")
    date_to_index = {}
    ret_wide = None

# === Factor Export Helper Function ===
def export_factor(factor_wide, factor_name, description):
    """Helper function to export factors in consistent format"""
    try:
        # Ensure datetime index
        if not isinstance(factor_wide.index, pd.DatetimeIndex):
            factor_wide.index = pd.to_datetime(factor_wide.index)
        
        # Filter to common dates and map to indices
        aligned_factor = factor_wide[factor_wide.index.isin(date_to_index.keys())]
        aligned_factor.index = aligned_factor.index.map(date_to_index)
        aligned_factor = aligned_factor.sort_index()
        
        # Export
        output_path = f"{output_dir}/{factor_name}"
        aligned_factor.to_csv(output_path)
        
        print(f"✅ {description}")
        print(f"   📁 File: {factor_name}")
        print(f"   📊 Shape: {aligned_factor.shape}")
        print(f"   📈 Non-null ratio: {aligned_factor.count().sum()/(aligned_factor.shape[0]*aligned_factor.shape[1]):.2%}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error exporting {factor_name}: {e}")
        return False

print(f"\n🎯 Ready for systematic factor generation...")

In [ ]:
# === 4. Technical Factor 1: Realized Skewness ===

print("📊 Generating Technical Factor 1: Realized Skewness")

if result_df is not None:
    def custom_skew(x):
        """Calculate custom skewness measure for intraday returns"""
        n = x['close'].count()
        if n < 2:
            return np.nan
        numerator = (x['close'] ** 3).sum() * np.sqrt(n)
        denominator = ((x['close'] ** 2).sum()) ** 1.5
        return numerator / denominator if denominator != 0 else np.nan

    print("🔄 Computing realized skewness for each stock-day...")
    
    # Calculate skewness by stock and date
    skewed = result_df.groupby(['order_book_id', 'date']).apply(custom_skew)
    df = skewed.reset_index()
    df.columns = ['order_book_id', 'date', 'skewed'] 
    df = df[['date', 'order_book_id', 'skewed']]
    
    print(f"   ✅ Calculated skewness for {len(df)} stock-day combinations")
    
    # Convert to wide format
    df['date'] = pd.to_datetime(df['date'])
    factor_wide = df.pivot(index="date", columns="order_book_id", values="skewed")
    
    # Export factor
    success = export_factor(factor_wide, "realized_skewness_daily_data.csv", 
                          "Realized Skewness factor exported")
    
    if success:
        print(f"📈 Factor interpretation:")
        print(f"   - Positive skewness: Right-tailed intraday return distribution")
        print(f"   - Negative skewness: Left-tailed intraday return distribution")
        print(f"   - Usage: Capture asymmetric risk patterns in intraday trading")
        
        # Display sample statistics
        print(f"\n📊 Factor statistics:")
        print(f"   - Mean: {factor_wide.stack().mean():.6f}")
        print(f"   - Std: {factor_wide.stack().std():.6f}")
        print(f"   - Min: {factor_wide.stack().min():.6f}")
        print(f"   - Max: {factor_wide.stack().max():.6f}")
    
else:
    print("❌ Cannot generate realized skewness: No data available")

In [ ]:
# === 5. Time-Based Factor 1: Closing Session Volume Ratio ===

print("🕐 Generating Time-Based Factor 1: Closing Session Volume Ratio")

if result_df is not None:
    from datetime import time
    
    print("🔄 Computing closing session (14:00-15:30) volume concentration...")
    
    # Ensure datetime column is properly formatted
    result_df_temp = result_df.copy()
    result_df_temp['datetime'] = pd.to_datetime(result_df_temp['datetime'])
    result_df_temp['time_only'] = result_df_temp['datetime'].dt.time
    
    # Define closing session time window
    start_time = time(14, 0, 0)   # 2:00 PM
    end_time = time(15, 30, 0)    # 3:30 PM
    
    # Filter data for closing session
    filtered = result_df_temp[(result_df_temp['time_only'] >= start_time) & 
                             (result_df_temp['time_only'] <= end_time)]
    
    print(f"   📊 Closing session data: {len(filtered):,} records")
    
    # Calculate volume ratios
    numerator = filtered.groupby(['order_book_id', 'date'])['volume'].sum()
    denominator = result_df_temp.groupby(['order_book_id', 'date'])['volume'].sum()
    at_close_ratio = numerator / denominator
    
    print(f"   ✅ Calculated closing ratios for {len(at_close_ratio)} stock-day combinations")
    
    # Convert to DataFrame and wide format
    at_close_ratio.name = 'at_close_ratio'
    df = at_close_ratio.reset_index()
    df.columns = ['order_book_id', 'date', 'at_close_ratio'] 
    df = df[['date', 'order_book_id', 'at_close_ratio']]
    df['date'] = pd.to_datetime(df['date'])
    
    factor_wide = df.pivot(index="date", columns="order_book_id", values="at_close_ratio")
    
    # Export factor
    success = export_factor(factor_wide, "at_close_ratio_daily_data.csv", 
                          "Closing Session Volume Ratio factor exported")
    
    if success:
        print(f"📈 Factor interpretation:")
        print(f"   - High ratio: Heavy trading during closing session (institutional activity)")
        print(f"   - Low ratio: Even distribution throughout the day")
        print(f"   - Usage: Capture end-of-day trading patterns and institutional behavior")
        
        # Display sample statistics
        print(f"\n📊 Factor statistics:")
        print(f"   - Mean: {factor_wide.stack().mean():.4f}")
        print(f"   - Std: {factor_wide.stack().std():.4f}")
        print(f"   - Range: [{factor_wide.stack().min():.4f}, {factor_wide.stack().max():.4f}]")
    
else:
    print("❌ Cannot generate closing session ratio: No data available")

In [ ]:
# === 6. Comprehensive Alpha Factor Generation ===

print("🚀 Starting comprehensive alpha factor generation process...")

if result_df is not None:
    
    # === Time-Based Factor 2: Opening Session Volume Ratio ===
    print("\n🕐 Generating Opening Session Volume Ratio...")
    
    result_df_temp = result_df.copy()
    result_df_temp['datetime'] = pd.to_datetime(result_df_temp['datetime'])
    result_df_temp['time_only'] = result_df_temp['datetime'].dt.time
    
    start_time = time(10, 0, 0)   # 10:00 AM
    end_time = time(11, 30, 0)    # 11:30 AM
    
    filtered = result_df_temp[(result_df_temp['time_only'] >= start_time) & 
                             (result_df_temp['time_only'] <= end_time)]
    
    numerator = filtered.groupby(['order_book_id', 'date'])['volume'].sum()
    denominator = result_df_temp.groupby(['order_book_id', 'date'])['volume'].sum()
    at_open_ratio = numerator / denominator
    
    df = at_open_ratio.reset_index()
    df.columns = ['order_book_id', 'date', 'at_open_ratio']
    df['date'] = pd.to_datetime(df['date'])
    factor_wide = df.pivot(index="date", columns="order_book_id", values="at_open_ratio")
    
    export_factor(factor_wide, "at_open_ratio_daily_data.csv", 
                  "Opening Session Volume Ratio factor exported")
    
    # === Technical Factor 2: Price-Volume Correlation ===
    print("\n📈 Generating Price-Volume Correlation...")
    
    def fast_corr(close, volume):
        """Fast correlation calculation between price and volume"""
        valid = ~np.isnan(close) & ~np.isnan(volume)
        close = close[valid]
        volume = volume[valid]
        
        if len(close) < 2 or volume.sum() == 0:
            return np.nan

        volume = volume / volume.sum()
        
        close_mean = close.mean()
        volume_mean = volume.mean()

        numerator = np.sum((close - close_mean) * (volume - volume_mean))
        denominator = np.sqrt(np.sum((close - close_mean)**2) * np.sum((volume - volume_mean)**2))
        
        return numerator / denominator if denominator != 0 else np.nan

    grouped = result_df.groupby(['order_book_id', 'date'])
    records = []
    for (order_book_id, date), group in tqdm(grouped, desc="Computing price-volume correlations"):
        corr = fast_corr(group['close'].values, group['volume'].values)
        records.append((order_book_id, date, corr))

    daily_corr_df = pd.DataFrame(records, columns=['order_book_id', 'date', 'close_volume_corr'])
    daily_corr_df['date'] = pd.to_datetime(daily_corr_df['date'])
    factor_wide = daily_corr_df.pivot(index="date", columns="order_book_id", values="close_volume_corr")
    
    export_factor(factor_wide, "high_close_volume_corr_daily_data.csv", 
                  "Price-Volume Correlation factor exported")
    
    # === Volume Factor 1: Top 30% Amount-Weighted Returns ===
    print("\n💰 Generating Top 30% Amount-Weighted Returns...")
    
    def get_top30_factor_wide_fast(df):
        """Calculate top 30% amount-weighted return product"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['amount'] = df['close'] * df['volume']
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
        df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
        df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
        df['threshold'] = np.ceil(df['group_size'] * 0.3)
        
        top30_df = df[df['rank'] < df['threshold']]
        top30_df['product_component'] = 1 + top30_df['return_rate']
        grouped = top30_df.groupby(['order_book_id', 'date'])['product_component'].prod()
        
        factor_wide = grouped.unstack(level=0)
        return factor_wide

    factor_wide = get_top30_factor_wide_fast(result_df)
    export_factor(factor_wide, "top_30_prod_daily_data.csv", 
                  "Top 30% Amount-Weighted Returns factor exported")
    
    # === Volume Factor 2: Large Volume Clustering ===
    print("\n📊 Generating Large Volume Clustering...")
    
    def get_hug_amount_factor_wide(df):
        """Calculate concentration of large volume trades"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        
        def get_hug_amount(group):
            bar = group['volume'].mean()
            std = group['volume'].std()
            group1 = group[group['volume'] >= bar + std]
            return group1['volume'].sum() / group['volume'].sum()
        
        factor_series = df.groupby(['order_book_id', 'date']).apply(get_hug_amount)
        factor_wide = factor_series.reset_index().pivot(
            index='date', columns='order_book_id', values=0)
        return factor_wide

    factor_wide = get_hug_amount_factor_wide(result_df)
    export_factor(factor_wide, "hug_amount_percent_daily_data.csv", 
                  "Large Volume Clustering factor exported")
    
    # === Technical Factor 3: Intraday Volatility ===
    print("\n📈 Generating Intraday Volatility...")
    
    def get_intraday_volatility_factor_wide(df):
        """Calculate intraday return volatility"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        factor_series = df.groupby(['order_book_id', 'date'])['return_rate'].std()
        factor_wide = factor_series.reset_index().pivot(
            index='date', columns='order_book_id', values='return_rate')
        return factor_wide

    factor_wide = get_intraday_volatility_factor_wide(result_df)
    export_factor(factor_wide, "intraday_volatility_daily_data.csv", 
                  "Intraday Volatility factor exported")
    
    print(f"\n✅ Comprehensive factor generation completed!")
    print(f"📊 Generated 6 major alpha factors covering:")
    print(f"   - Time-based patterns (opening/closing sessions)")
    print(f"   - Technical indicators (skewness, volatility, correlations)")
    print(f"   - Volume microstructure (clustering, amount-weighted returns)")

else:
    print("❌ Cannot generate factors: No data available")

In [ ]:
# === 7. Advanced Statistical Factors Generation ===

print("🧮 Generating Advanced Statistical Factors...")

if result_df is not None:
    
    # === Advanced Factor 1: Top 20% Amount-Weighted Returns ===
    print("\n💰 Generating Top 20% Amount-Weighted Returns...")
    
    def get_top20_factor_wide_fast(df):
        """Calculate top 20% amount-weighted return product"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['amount'] = df['close'] * df['volume']
        df['return_rate'] = (df['close'] - df['open']) / df['open']
        
        df.sort_values(['order_book_id', 'date', 'amount'], ascending=[True, True, False], inplace=True)
        df['rank'] = df.groupby(['order_book_id', 'date']).cumcount()
        df['group_size'] = df.groupby(['order_book_id', 'date'])['amount'].transform('count')
        df['threshold'] = np.ceil(df['group_size'] * 0.2)
        
        top20_df = df[df['rank'] < df['threshold']]
        top20_df['product_component'] = 1 + top20_df['return_rate']
        grouped = top20_df.groupby(['order_book_id', 'date'])['product_component'].prod()
        
        factor_wide = grouped.unstack(level=0)
        return factor_wide

    factor_wide = get_top20_factor_wide_fast(result_df)
    export_factor(factor_wide, "top_20_prod_daily_data.csv", 
                  "Top 20% Amount-Weighted Returns factor exported")
    
    # === Advanced Factor 2: Volume-Weighted Close Factor ===
    print("\n📊 Generating Volume-Weighted Close Factor...")
    
    def get_weighted_close(df):
        """Calculate volume-weighted close factor"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['weighted_value'] = df['close'] * df['volume']
        df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('mean')
        df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
        df['weighted_close'] = df['weighted_value'] / (df['vol'] * df['d2'])
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='weighted_close')
        return factor_wide

    factor_wide = get_weighted_close(result_df)
    export_factor(factor_wide, "weighted_close_daily_data.csv", 
                  "Volume-Weighted Close factor exported")
    
    # === Advanced Factor 3: Information Entropy ===
    print("\n🔬 Generating Information Entropy Factor...")
    
    def get_entropy(df):
        """Calculate information entropy based on volume-price distribution"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['d2'] = df.groupby(['order_book_id', 'date'])['close'].transform('sum')
        df['vol'] = df.groupby(['order_book_id', 'date'])['volume'].transform('sum')
        df['single'] = (df['volume'] / df['vol']) * (df['close'] / df['d2'])
        df['d3'] = -df['single'] * np.log(df['single'] + 1e-10)  # Add small epsilon to avoid log(0)
        df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('sum')
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_entropy(result_df)
    export_factor(factor_wide, "entropy_daily_weighted_close_data.csv", 
                  "Information Entropy factor exported")
    
    # === Advanced Factor 4: Volume Difference Statistics ===
    print("\n📈 Generating Volume Difference Statistics...")
    
    def get_diff_std(df):
        """Calculate volume difference standard deviation"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
        df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
        df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
        df['d1'] = df['nom1'] / df['nom2']
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_diff_std(result_df)
    export_factor(factor_wide, "diff_std_data.csv", 
                  "Volume Difference Std factor exported")
    
    def get_diff_mean(df):
        """Calculate volume difference mean"""
        df = df.copy()
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['volume'].diff()
        df['nom2'] = df.groupby(['order_book_id', 'date'])['volume'].transform('mean')
        df['d3'] = np.abs(df['diff'] / df['nom2'])
        df['d1'] = df.groupby(['order_book_id', 'date'])['d3'].transform('mean')
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_diff_mean(result_df)
    export_factor(factor_wide, "diff_mean_data.csv", 
                  "Volume Difference Mean factor exported")
    
    # === Advanced Factor 5: Amount-based Difference Statistics ===
    print("\n💱 Generating Amount-based Difference Statistics...")
    
    def get_amount_diff_std(df):
        """Calculate amount difference standard deviation"""
        df = df.copy()
        df['amount'] = df['volume'] * df['close']
        df['datetime'] = pd.to_datetime(df['datetime'])
        df['date'] = df['datetime'].dt.date
        df['diff'] = df.groupby(['order_book_id', 'date'])['amount'].diff()
        df['nom1'] = df.groupby(['order_book_id', 'date'])['diff'].transform('std')
        df['nom2'] = df.groupby(['order_book_id', 'date'])['amount'].transform('mean')
        df['d1'] = df['nom1'] / df['nom2']
        
        factor_wide = df.pivot_table(index='date', columns='order_book_id', values='d1')
        return factor_wide

    factor_wide = get_amount_diff_std(result_df)
    export_factor(factor_wide, "diff_std_amount_data.csv", 
                  "Amount Difference Std factor exported")
    
    print(f"\n✅ Advanced statistical factors generation completed!")
    print(f"📊 Generated 7 additional sophisticated factors")

else:
    print("❌ Cannot generate advanced factors: No data available")

In [ ]:
# === 8. Final Summary and Factor Library Report ===

print("="*80)
print("📋 ALPHA FACTOR GENERATION 2 - SUMMARY REPORT")
print("="*80)

print(f"\n🎯 METHODOLOGY:")
print(f"   Data Source: High-frequency intraday minute-level OHLCV data")
print(f"   Processing: Combined datamin2 and datamin3 datasets")
print(f"   Approach: Technical & microstructure-based factor engineering")
print(f"   Output Format: Wide format (dates × stocks) aligned with return data")

print(f"\n📊 GENERATED FACTOR LIBRARY:")

factor_categories = {
    "🕐 Time-Based Factors": [
        "at_close_ratio_daily_data.csv - Closing session volume concentration",
        "at_open_ratio_daily_data.csv - Opening session volume concentration"
    ],
    "📈 Technical & Statistical Factors": [
        "realized_skewness_daily_data.csv - Intraday return distribution asymmetry",
        "high_close_volume_corr_daily_data.csv - Price-volume correlation",
        "intraday_volatility_daily_data.csv - Intraday return volatility"
    ],
    "💰 Volume & Amount Factors": [
        "top_30_prod_daily_data.csv - Top 30% amount-weighted return product",
        "top_20_prod_daily_data.csv - Top 20% amount-weighted return product",
        "hug_amount_percent_daily_data.csv - Large volume trade clustering"
    ],
    "🧮 Advanced Statistical Factors": [
        "weighted_close_daily_data.csv - Volume-weighted close deviation",
        "entropy_daily_weighted_close_data.csv - Information entropy measure",
        "diff_std_data.csv - Volume difference standard deviation",
        "diff_mean_data.csv - Volume difference mean",
        "diff_std_amount_data.csv - Amount difference standard deviation"
    ]
}

total_factors = 0
for category, factors in factor_categories.items():
    print(f"\n{category}:")
    for factor in factors:
        print(f"   • {factor}")
        total_factors += 1

print(f"\n📁 OUTPUT LOCATION:")
print(f"   Directory: ../data/factors/obtained_features/")
print(f"   Total Factors Generated: {total_factors}")
print(f"   Format: CSV files with date indices aligned to return data")

print(f"\n🔬 RESEARCH APPLICATIONS:")
print(f"   • Market Microstructure Analysis: Time-based and volume clustering patterns")
print(f"   • Technical Analysis: Statistical moments and price-volume relationships") 
print(f"   • Risk Management: Volatility and distribution asymmetry measures")
print(f"   • Alpha Generation: Sophisticated quantile-based and entropy factors")

print(f"\n🎯 NEXT STEPS:")
print(f"   1. Run factor backtesting using factor_backtest.ipynb")
print(f"   2. Evaluate IC and performance metrics for each factor")
print(f"   3. Include promising factors in Alpha_Factor_Selection.ipynb")
print(f"   4. Test factor combinations and portfolio construction")
print(f"   5. Optimize parameters for different market regimes")

print(f"\n🚀 HIGH-FREQUENCY ALPHA FACTOR GENERATION COMPLETED!")
print(f"   Ready for systematic evaluation and strategy implementation")

print(f"\n" + "="*80)